# Local DuckDB Scratch Starter

Minimal notebook starter for opening a PILATES archive while copying the Consist DuckDB file to local scratch first.

Use this when the archive lives on shared scratch or NFS but notebook queries should read from node-local storage.


## 1) Configure paths

Set `ARCHIVE_RUN_DIR` to the completed PILATES archive run directory. The notebook will find the Consist DB under `.consist/`, copy it to `LOCAL_SCRATCH`, and open the archive with `db_path=LOCAL_DB_PATH`.


In [ ]:
from pathlib import Path
import os
import shutil
import time

from pilates_consist_analysis import open_archive

ARCHIVE_RUN_DIR = Path('/path/to/archive/run')
PROJECT_ROOT = Path('/Users/zaneedell/git/PILATES')

# On Lawrencium this defaults to /local/job$SLURM_JOB_ID/pilates-analysis.
# Outside Slurm, set PILATES_ANALYSIS_SCRATCH or edit this path explicitly.
_slurm_job_id = os.environ.get('SLURM_JOB_ID', 'manual')
LOCAL_SCRATCH = Path(os.environ.get('PILATES_ANALYSIS_SCRATCH', f'/local/job{_slurm_job_id}/pilates-analysis'))
LOCAL_DB_PATH = LOCAL_SCRATCH / 'pilates-analysis.duckdb'

ARCHIVE_RUN_DIR, LOCAL_DB_PATH

## 2) Resolve the archive DB

These are the same default DB locations used by the analysis package when `db_path` is not provided.


In [ ]:
DB_CANDIDATES = (
    Path('.consist/snapshots/latest/provenance.duckdb'),
    Path('.consist/provenance.duckdb'),
    Path('.consist/snapshots/latest/consist.duckdb'),
    Path('.consist/consist.duckdb'),
)


def resolve_archive_db(archive_run_dir: Path) -> Path:
    for relative_path in DB_CANDIDATES:
        candidate = archive_run_dir / relative_path
        if candidate.exists():
            return candidate.resolve()
    searched = '\n  - '.join(str(archive_run_dir / p) for p in DB_CANDIDATES)
    raise FileNotFoundError(f'No Consist DB found. Checked:\n  - {searched}')


SOURCE_DB_PATH = resolve_archive_db(ARCHIVE_RUN_DIR)
SOURCE_DB_PATH

## 3) Copy the DB to local scratch

The helper below checks that the source DB size is stable before copying. If a WAL sidecar exists, it copies that too. Run this only after the PILATES job has finished or when you know the DB is not being written.


In [ ]:
def _file_size(path: Path) -> int:
    return path.stat().st_size


def copy_duckdb_to_local_scratch(
    source_db: Path,
    destination_db: Path,
    *,
    sleep_seconds: float = 2.0,
) -> Path:
    source_db = source_db.expanduser().resolve()
    destination_db = destination_db.expanduser()
    if not source_db.is_file():
        raise FileNotFoundError(f'Source DB not found: {source_db}')

    size_before = _file_size(source_db)
    time.sleep(sleep_seconds)
    size_after = _file_size(source_db)
    if size_before != size_after:
        raise RuntimeError(
            'Source DB appears to be changing. Wait for the run to finish or copy a snapshot.'
        )

    destination_db.parent.mkdir(parents=True, exist_ok=True)
    tmp_db = destination_db.with_name(f'{destination_db.name}.tmp.{os.getpid()}')
    shutil.copy2(source_db, tmp_db)

    source_wal = source_db.with_suffix(source_db.suffix + '.wal')
    tmp_wal = tmp_db.with_suffix(tmp_db.suffix + '.wal')
    destination_wal = destination_db.with_suffix(destination_db.suffix + '.wal')
    if source_wal.exists():
        shutil.copy2(source_wal, tmp_wal)

    tmp_db.replace(destination_db)
    if tmp_wal.exists():
        tmp_wal.replace(destination_wal)

    return destination_db.resolve()


LOCAL_DB_PATH = copy_duckdb_to_local_scratch(SOURCE_DB_PATH, LOCAL_DB_PATH)
LOCAL_DB_PATH

## 4) Open the archive using the local DB copy

The archive mount still points at `ARCHIVE_RUN_DIR`, so artifact paths resolve against the preserved archive. Only the Consist DB reads go to the local copy.


In [ ]:
archive = open_archive(
    ARCHIVE_RUN_DIR,
    project_root=PROJECT_ROOT,
    db_path=LOCAL_DB_PATH,
)

display(archive.summary())
archive.issues()

## 5) First discovery checks

These are intentionally small. Once this works, switch to `archive_exploration_starter.ipynb` or your project notebook for deeper analysis.


In [ ]:
print('Scenarios:', archive.scenarios())
print('Years:', archive.years())
print('Models:', archive.models())

display(archive.runs().head(20))

## Shell alternative

From a Lawrencium shell, the repo helper performs the same safe copy pattern:

```bash
hpc/copy_duckdb_local.sh --src /path/to/archive/.consist/provenance.duckdb
```

It prints the `PILATES_DB_PATH` value you can use as `LOCAL_DB_PATH` in this notebook.
